# FibRec, FibMemo and FibIt (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_cpp.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `FibRec`, `FibMemo` and `FibIt`: the algorithms in C++, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [1]:
%%writefile fibonacci.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long fibCalls = 0;   // calls executed

// FibRec(n)
long long FibRec(int n)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    return FibRec(n - 1) + FibRec(n - 2);
}

// FibMemo(n, memo): memo[k] = -1 means "undefined"
long long FibMemo(int n, long long *memo)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    if (memo[n] == -1)
    {
        memo[n] = FibMemo(n - 1, memo) + FibMemo(n - 2, memo);
    }
    return memo[n];
}

// FibIt(n)
long long FibIt(int n)
{
    if (n <= 1)
    {
        return n;
    }
    long long prev = 0;             // F(i-2)
    long long curr = 1;             // F(i-1)
    int i = 2;
    while (i <= n)
    {
        long long next = prev + curr;
        prev = curr;
        curr = next;
        i = i + 1;
    }
    return curr;
}

// Like FibRec, but counts the calls for each value of the parameter
long long FibRecHist(int n, int *counts)
{
    counts[n]++;
    if (n <= 1)
    {
        return n;
    }
    return FibRecHist(n - 1, counts) + FibRecHist(n - 2, counts);
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int counts[6] = {0, 0, 0, 0, 0, 0};
    long long f5 = FibRecHist(5, counts);
    printf("FibRec(5) = %lld\n", f5);
    string line;
    int total = 0;
    for (int k = 5; k >= 0; k--)
    {
        if (k < 5) line += "  ";
        line += to_string(k) + ":" + to_string(counts[k]);
        total += counts[k];
    }
    printf("calls for each value of the parameter: %s   total %d\n", line.c_str(), total);
    printf("\n");
    printf("%3s %8s %9s %10s %8s %6s  (calls, calls, calls, iterations)\n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
    int ns[] = {10, 15, 20, 25, 30};
    for (int n : ns)
    {
        long long f = FibIt(n);
        fibCalls = 0;
        long long r1 = FibRec(n);
        long long c1 = fibCalls;
        fibCalls = 0;
        vector<long long> memo(n + 1, -1);
        long long r2 = FibMemo(n, memo.data());
        long long c2 = fibCalls;
        bool ok = (r1 == r2 && r2 == f && c1 == 2 * FibIt(n + 1) - 1 && c2 == 2 * n - 1);
        printf("%3d %8lld %9lld %10lld %8lld %6d  %s\n", n, f, c1, 2 * FibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (int n : {20, 25, 30})
    {
        double t0 = NowMs(); volatile long long sink = FibRec(n); double t1 = NowMs() - t0;
        t0 = NowMs();
        for (int rep = 0; rep < 1000; rep++)
        {
            sink = FibIt(n);
        }
        double t2 = (NowMs() - t0) / 1000;
        printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms\n", n, t1, t2);
    }
    return 0;
}
#endif

Writing fibonacci.cpp


In [2]:
!g++ -std=c++17 -O2 -o fibonacci fibonacci.cpp && ./fibonacci

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     0.0204 ms   FibIt   0.000012 ms
n=25  FibRec     0.2659 ms   FibIt   0.000014 ms
n=30  FibRec     2.5816 ms   FibIt   0.000018 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [3]:
%%writefile prova_fibonacci.cpp
#define NOMAIN
#include "fibonacci.cpp"

int main()
{
    // Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
    int n = 32;
    fibCalls = 0;
    long long r1 = FibRec(n);
    printf("%lld %lld\n", r1, fibCalls);
    fibCalls = 0;
    vector<long long> memo(n + 1, -1);
    long long r2 = FibMemo(n, memo.data());
    printf("%lld %lld\n", r2, fibCalls);
    return 0;
}

Writing prova_fibonacci.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fibonacci prova_fibonacci.cpp && ./prova_fibonacci

2178309 7049155
2178309 63


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).